In [16]:
print("Check_Python")
%load_ext autotime

Check_Python
The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 15 ms (started: 2026-03-26 00:43:36 +05:30)


In [17]:
import tensorflow as tf

import os
from pathlib import Path

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau

import numpy as np

time: 0 ns (started: 2026-03-26 00:43:36 +05:30)


In [18]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
time: 0 ns (started: 2026-03-26 00:43:36 +05:30)


In [19]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists())

True True True
time: 78 ms (started: 2026-03-26 00:43:36 +05:30)


In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"
SEED = 42

time: 0 ns (started: 2026-03-25 23:39:46 +05:30)


In [6]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

time: 0 ns (started: 2026-03-25 23:39:46 +05:30)


In [9]:
val_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

time: 0 ns (started: 2026-03-25 23:47:12 +05:30)


In [10]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale'
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale'
)

Found 4679 images belonging to 2 classes.
Found 553 images belonging to 2 classes.
time: 531 ms (started: 2026-03-25 23:47:13 +05:30)


In [11]:
labels = train_generator.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))

print("Class indices:", train_generator.class_indices)
print("Class distribution:", np.bincount(labels))
print("Original Class weights:", class_weights)

class_weights = {
    0: 1.6,   # NORMAL (was ~1.95)
    1: 0.8    # PNEUMONIA (was ~0.67)
}

print("Adjusted Class weights:", class_weights)

Class indices: {'NORMAL': 0, 'PNEUMONIA': 1}
Class distribution: [1199 3480]
Original Class weights: {0: np.float64(1.951209341117598), 1: np.float64(0.6722701149425288)}
Adjusted Class weights: {0: 1.6, 1: 0.8}
time: 47 ms (started: 2026-03-25 23:47:13 +05:30)


In [12]:
from tensorflow.keras.optimizers import Adam

def build_model():
    m = models.Sequential([
        layers.Input(shape=(224, 224, 1)),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.1),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        # Block 4
        layers.Conv2D(256, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.3),

        layers.GlobalAveragePooling2D(),

        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])

    m.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return m

time: 0 ns (started: 2026-03-25 23:47:13 +05:30)


layers.GlobalAveragePooling2D() :-
To reduce parameters and prevent overfitting by summarizing each feature map instead of flattening everything

In [13]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        "best_model_class_weights.h5",
        monitor="val_loss",
        save_best_only=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

time: 0 ns (started: 2026-03-25 23:47:14 +05:30)


In [14]:
model = build_model()
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 941ms/step - accuracy: 0.6953 - loss: 0.8889

147/147 ━━━━━━━━━━━━━━━━━━━━ 151s 991ms/step - accuracy: 0.7756 - loss: 0.7756 - val_accuracy: 0.7288 - val_loss: 1.7649 - learning_rate: 1.0000e-04
Epoch 2/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 140s 950ms/step - accuracy: 0.8818 - loss: 0.5951 - val_accuracy: 0.7288 - val_loss: 2.8248 - learning_rate: 1.0000e-04
Epoch 3/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 140s 951ms/step - accuracy: 0.9006 - loss: 0.5316 - val_accuracy: 0.7288 - val_loss: 3.3885 - learning_rate: 1.0000e-04
Epoch 4/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 142s 964ms/step - accuracy: 0.9098 - loss: 0.5121 - val_accuracy: 0.7288 - val_loss: 2.3699 - learning_rate: 3.0000e-05
Epoch 5/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 918ms/step - accuracy: 0.9150 - loss: 0.4865

147/147 ━━━━━━━━━━━━━━━━━━━━ 141s 958ms/step - accuracy: 0.9113 - loss: 0.4945 - val_accuracy: 0.7306 - val_loss: 1.3377 - learning_rate: 3.0000e-05
Epoch 6/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 939ms/step - accuracy: 0.9187 - loss: 0.4662

147/147 ━━━━━━━━━━━━━━━━━━━━ 144s 980ms/step - accuracy: 0.9154 - loss: 0.4847 - val_accuracy: 0.8065 - val_loss: 0.6715 - learning_rate: 3.0000e-05
Epoch 7/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 915ms/step - accuracy: 0.9135 - loss: 0.4839

147/147 ━━━━━━━━━━━━━━━━━━━━ 141s 954ms/step - accuracy: 0.9175 - loss: 0.4815 - val_accuracy: 0.9150 - val_loss: 0.4771 - learning_rate: 3.0000e-05
Epoch 8/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 141s 957ms/step - accuracy: 0.9149 - loss: 0.4850 - val_accuracy: 0.8535 - val_loss: 0.5525 - learning_rate: 3.0000e-05
Epoch 9/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 140s 953ms/step - accuracy: 0.9218 - loss: 0.4583 - val_accuracy: 0.7396 - val_loss: 1.2148 - learning_rate: 3.0000e-05
Epoch 10/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 142s 961ms/step - accuracy: 0.9179 - loss: 0.4642 - val_accuracy: 0.8933 - val_loss: 0.4972 - learning_rate: 9.0000e-06
Epoch 11/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 141s 955ms/step - accuracy: 0.9269 - loss: 0.4430 - val_accuracy: 0.8770 - val_loss: 0.5260 - learning_rate: 9.0000e-06
Epoch 12/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 142s 965ms/step - accuracy: 0.9228 - loss: 0.4505 - val_accuracy: 0.8951 - val_loss: 0.4946 - learning_rate: 2.7000e-06
time: 28min 27s (started: 2026-03-25 23:47:47 +0

In [15]:
print("End")

End
time: 0 ns (started: 2026-03-26 00:16:15 +05:30)
